# 10_01 — HOG + Approximate RBF SVM Baseline (Classical ML on Handcrafted Features)

## Goal
Train and benchmark a classical baseline model using **Histogram of Oriented Gradients (HOG)** features and an **approximate RBF-kernel SVM** implemented as:

StandardScaler → Nyström RBF feature map → LinearSVC

This approach preserves the **non-linear decision boundary behavior of an RBF SVM** while dramatically reducing training time compared to an exact `SVC(kernel="rbf")`, making it feasible to train on the full dataset.

The experiment runs on the fixed dataset split **split_v1** created in Phase 1.

This notebook produces:

- A fully trained scikit-learn pipeline  
  `StandardScaler → Nystroem(RBF) → LinearSVC`
- Reproducible evaluation metrics  
  (accuracy, macro-F1, precision, recall, confusion matrix)
- Serialized artifacts under  
  `models/ml_basic_features/hog_svm/run_YYYYMMDD_HHMMSS/`
- A metrics summary under  
  `reports/metrics/hog_svm_svm_rbf_approx_nystroem_linearsvc_metrics.json`
- Experiment tracking in **MLflow** under the project tracking directory  
  `mlruns/`

This notebook is designed so **multiple experiments can run in parallel safely** by generating unique run IDs.

---

# Data Contract (Reusing Phase 1 Infrastructure)

Dataset splits created in Phase 1 are reused without modification.

Loaded from:

- `data/splits/split_v1/train.csv`
- `data/splits/split_v1/val.csv`
- `data/splits/split_v1/test.csv`

Class mapping:

- `data/splits/split_v1/classes.json`

No dataset files are modified during this notebook.

---

# Feature Pipeline (Cached HOG Features)

HOG features were already extracted and cached earlier in Phase 2 to avoid recomputation.

Cached feature location:


data/processed/features/split_v1/hog/


Files used:


train.npy
val.npy
test.npy

labels_train.npy
labels_val.npy
labels_test.npy


These contain **precomputed HOG feature vectors for the entire dataset**.

As a result, this notebook **skips feature extraction entirely** and directly loads cached features for training.

Legacy feature extraction code remains in the notebook (commented) for reproducibility and documentation.

---

# Image Preprocessing for HOG

For handcrafted features we intentionally compute gradients from **pixel-space images**, not normalized tensors used in deep learning pipelines.

Image preprocessing used during feature extraction:

- Load image using PIL in RGB
- Resize to fixed resolution **224×224**
- Convert to grayscale
- Compute HOG via `skimage.feature.hog`

HOG parameters:

- orientations = 9  
- pixels_per_cell = (8, 8)  
- cells_per_block = (2, 2)  
- block_norm = L2-Hys  

This configuration captures local gradient structure suitable for classical vision models.

---

# Model Architecture

Classifier pipeline:


StandardScaler
↓
Nyström RBF feature approximation
↓
LinearSVC


Components:

**StandardScaler**

Normalizes feature magnitudes to stabilize training.

**Nyström RBF Approximation**

Approximates the RBF kernel by mapping features into a higher-dimensional space.  
This provides non-linear decision boundaries similar to an RBF SVM while scaling linearly with dataset size.

**LinearSVC**

Efficient linear SVM trained on the approximated RBF feature space.

This architecture preserves the **spirit of an RBF SVM** while reducing training time from potentially many hours to a manageable runtime.

---

# Outputs

Artifacts saved per run:


models/ml_basic_features/hog_svm/run_YYYYMMDD_HHMMSS/
model.pkl
config.json
metrics.json


Metrics report:


reports/metrics/hog_svm_svm_rbf_approx_nystroem_linearsvc_metrics.json


MLflow experiment tracking:


mlruns/


Logged information:

- model parameters
- feature configuration
- classifier parameters
- evaluation metrics
- trained model artifact

---

# Verification

At the end of the notebook:

1. The serialized model is reloaded from disk.
2. A small batch of samples is passed through `predict`.
3. Predicted class labels are printed.

This confirms that:

- the model was saved correctly
- the pipeline loads successfully
- inference works as expected.

In [ ]:
# Cell 1

import os
import json
import pickle
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

from skimage.feature import hog

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import Nystroem  # fast RBF-ish approximation
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

import mlflow

In [ ]:
# Cell 2 
# Robust project root detection for notebooks (works even if data/ isn't present yet)
NOTEBOOK_DIR = Path.cwd().expanduser().resolve()

def find_project_root(start: Path) -> Path:
    markers_any = [
        ".git",              # repo root
        "pyproject.toml",    # common project root marker
        "requirements.txt",  # common project root marker
    ]
    markers_all = [
        "src",
        "configs",
    ]

    for p in [start] + list(start.parents):
        has_all = all((p / m).exists() for m in markers_all)
        has_any = any((p / m).exists() for m in markers_any)
        if has_all and has_any:
            return p

    # Fallback: if repo doesn't have .git (e.g., zipped project), accept src+configs only
    for p in [start] + list(start.parents):
        if all((p / m).exists() for m in markers_all):
            return p

    raise RuntimeError(
        f"Could not locate project root from: {start}\n"
        f"Expected to find at least: {markers_all} and (one of {markers_any}) in a parent directory."
    )

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

# Create expected directories if missing (safe even if already exists)
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_ID = "split_v1"
SPLITS_DIR = DATA_DIR / "splits" / SPLIT_ID

TRAIN_CSV = SPLITS_DIR / "train.csv"
VAL_CSV = SPLITS_DIR / "val.csv"
TEST_CSV = SPLITS_DIR / "test.csv"
CLASSES_JSON = SPLITS_DIR / "classes.json"

MODEL_NAME = "hog_svm"
FEATURE_TYPE = "hog"
CLASSIFIER_NAME = "svm_rbf_approx_nystroem_linearsvc"
TRANSFORM_ID = "pixelspace_resize224_grayscale_hog"

RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")

RUN_DIR = MODELS_DIR / "ml_basic_features" / MODEL_NAME / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_METRICS_DIR = REPORTS_DIR / "metrics"
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)

REPORT_METRICS_PATH = REPORTS_METRICS_DIR / f"{MODEL_NAME}_{CLASSIFIER_NAME}_metrics.json"

FEATURES_DIR = DATA_DIR / "processed" / "features" / SPLIT_ID / "hog"

print("NOTEBOOK_DIR :", NOTEBOOK_DIR)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("SPLITS_DIR   :", SPLITS_DIR)
print("FEATURES_DIR :", FEATURES_DIR)

NOTEBOOK_DIR : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/notebooks/10_ml_basic_features
PROJECT_ROOT : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server
SPLITS_DIR   : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/data/splits/split_v1
FEATURES_DIR : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/data/processed/features/split_v1/hog


In [ ]:
# Cell 3
TRACKING_DIR = PROJECT_ROOT / "mlruns"
TRACKING_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/05 15:22:33 INFO mlflow.tracking.fluent: Experiment with name 'AnimalClassification' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/mlruns/511423962448650871', creation_time=1772709753829, experiment_id='511423962448650871', last_update_time=1772709753829, lifecycle_stage='active', name='AnimalClassification', tags={}, workspace='default'>

## Loading split manifests

We load `train.csv`, `val.csv`, `test.csv` generated in Phase 1 (deterministic stratified split with seed=42).  
The CSV files contain `filepath,label`. We also load `classes.json` to map class names to integer ids consistently across experiments.

In [ ]:
# Cell 4
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    class_to_idx = json.load(f)

idx_to_class = {v: k for k, v in class_to_idx.items()}

train_df["label_idx"] = train_df["label"].map(class_to_idx)
val_df["label_idx"] = val_df["label"].map(class_to_idx)
test_df["label_idx"] = test_df["label"].map(class_to_idx)

assert train_df["label_idx"].isna().sum() == 0
assert val_df["label_idx"].isna().sum() == 0
assert test_df["label_idx"].isna().sum() == 0

train_df.head()

,filepath,label,label_idx
0,f:/Projects/AnimalClassification/data/prepared...,wildlife,2
1,f:/Projects/AnimalClassification/data/prepared...,dogs,1
2,f:/Projects/AnimalClassification/data/prepared...,dogs,1
3,f:/Projects/AnimalClassification/data/prepared...,wildlife,2
4,f:/Projects/AnimalClassification/data/prepared...,cats,0


In [ ]:
# Cell 5
IMG_SIZE = (224, 224)

HOG_PARAMS = {
    "orientations": 9,
    "pixels_per_cell": (8, 8),
    "cells_per_block": (2, 2),
    "block_norm": "L2-Hys",
    "transform_sqrt": True,
    "feature_vector": True
}

def load_image_resize_rgb(path):
    img = Image.open(path).convert("RGB")
    img = img.resize(IMG_SIZE)
    return img

def hog_features_from_path(path):
    img = load_image_resize_rgb(path)
    gray = np.array(img.convert("L"))
    feat = hog(gray, **HOG_PARAMS)
    return feat.astype(np.float32)

In [ ]:
# def build_features(df):
#     X = np.vstack([hog_features_from_path(p) for p in df["filepath"].values])
#     y = df["label_idx"].values.astype(np.int64)
#     return X, y

# X_train, y_train = build_features(train_df)
# X_val, y_val = build_features(val_df)
# X_test, y_test = build_features(test_df)

# X_train.shape, X_val.shape, X_test.shape

In [ ]:
# Cell 6 - Feature cache auto-detect + load/extract

def cache_paths(base_dir: Path):
    return {
        "train": base_dir / "train.npy",
        "val": base_dir / "val.npy",
        "test": base_dir / "test.npy",
        "labels_train": base_dir / "labels_train.npy",
        "labels_val": base_dir / "labels_val.npy",
        "labels_test": base_dir / "labels_test.npy",
    }

paths = cache_paths(FEATURES_DIR)
cache_exists = all(p.exists() for p in paths.values())

if cache_exists:
    print(f"[CACHE] Found cached features at: {FEATURES_DIR}")
    X_train = np.load(paths["train"], mmap_mode="r")
    X_val = np.load(paths["val"], mmap_mode="r")
    X_test = np.load(paths["test"], mmap_mode="r")

    y_train = np.load(paths["labels_train"])
    y_val = np.load(paths["labels_val"])
    y_test = np.load(paths["labels_test"])
else:
    print(f"[CACHE MISS] Extracting HOG features and saving to: {FEATURES_DIR}")
    FEATURES_DIR.mkdir(parents=True, exist_ok=True) # Ensure dir exists before saving

    def build_features(df: pd.DataFrame):
        X_list = []
        valid_indices = []

        for idx, p in enumerate(df["filepath"].values):
            # Fix Windows absolute paths for Ubuntu compatibility
            path_str = str(p).replace("\\", "/")
            if "data/prepared" in path_str:
                rel_path = path_str.split("data/prepared/")[1]
                true_path = PROJECT_ROOT / "data" / "prepared" / rel_path
            else:
                true_path = Path(p)

            try:
                feat = hog_features_from_path(str(true_path))
                X_list.append(feat)
                valid_indices.append(idx)
            except Exception as e:
                print(f"[WARNING] Skipping {true_path} - {e}")

        X = np.vstack(X_list)
        y = df["label_idx"].values[valid_indices].astype(np.int64)
        return X, y

    X_train, y_train = build_features(train_df)
    X_val, y_val = build_features(val_df)
    X_test, y_test = build_features(test_df)

    # Save features
    np.save(paths["train"], X_train.astype(np.float32))
    np.save(paths["val"], X_val.astype(np.float32))
    np.save(paths["test"], X_test.astype(np.float32))

    np.save(paths["labels_train"], y_train)
    np.save(paths["labels_val"], y_val)
    np.save(paths["labels_test"], y_test)

    print("[CACHE] Saved feature arrays.")

print("X_train:", X_train.shape, X_train.dtype)
print("X_val:", X_val.shape, X_val.dtype)
print("X_test:", X_test.shape, X_test.dtype)
print("y_train:", y_train.shape, y_train.dtype, "unique:", np.unique(y_train))

assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0] == y_val.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert set(np.unique(y_train)).issubset(set(class_to_idx.values()))

# Quick finite check on a small slice (fast)
assert np.isfinite(np.asarray(X_train[:1000])).all()

X_train: (50127, 26244) float32
X_val: (6266, 26244) float32
X_test: (6266, 26244) float32
y_train: (50127,) int64 unique: [0 1 2]


In [ ]:
# Cell 7
# Approximate-RBF SVM: Nyström (RBF feature map) + LinearSVC
# Much faster than exact RBF SVC on 50k samples, while keeping nonlinear "RBF spirit".

N_COMPONENTS = 5000  # try 5000 first; if fast, consider 8000 or 10000
GAMMA = 1.0 / X_train.shape[1]  # decent proxy for "scale" on standardized features

NYSTROEM_PARAMS = {
    "kernel": "rbf",
    "gamma": float(GAMMA),
    "n_components": int(N_COMPONENTS),
    "random_state": 42,
}

LINEARSVC_PARAMS = {
    "C": 1.0,
    "random_state": 42,
    "dual": "auto",
    "max_iter": 5000,
}

model = Pipeline([
    ("scaler", StandardScaler(with_mean=False, with_std=True)),
    ("rbf_map", Nystroem(**NYSTROEM_PARAMS)),
    ("clf", LinearSVC(**LINEARSVC_PARAMS)),
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('rbf_map', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",False
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"kernel kernel: str or callable, default='rbf'Kernel map to be approximated. A callable should accept two argumentsand the keyword arguments passed to this object as `kernel_params`, andshould return a floating point number.",'rbf'
,"gamma gamma: float, default=NoneGamma parameter for the RBF, laplacian, polynomial, exponential chi2and sigmoid kernels. Interpretation of the default value is left tothe kernel; see the documentation for sklearn.metrics.pairwise.Ignored by other kernels.",3.8103947568968146e-05
,"coef0 coef0: float, default=NoneZero coefficient for polynomial and sigmoid kernels.Ignored by other kernels.",None
,"degree degree: float, default=NoneDegree of the polynomial kernel. Ignored by other kernels.",None


In [ ]:
# from pathlib import Path
# import numpy as np

# FEATURES_DIR = DATA_DIR / "processed" / "features" / SPLIT_ID / "hog"
# FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# np.save(FEATURES_DIR / "train.npy", X_train)
# np.save(FEATURES_DIR / "val.npy", X_val)
# np.save(FEATURES_DIR / "test.npy", X_test)

# np.save(FEATURES_DIR / "labels_train.npy", y_train)
# np.save(FEATURES_DIR / "labels_val.npy", y_val)
# np.save(FEATURES_DIR / "labels_test.npy", y_test)

# str(FEATURES_DIR)

In [ ]:
# Cell 8
def evaluate(model, X, y):
    preds = model.predict(X)
    metrics = {
        "accuracy": float(accuracy_score(y, preds)),
        "f1_macro": float(f1_score(y, preds, average="macro")),
        "precision_macro": float(precision_score(y, preds, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y, preds, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(y, preds).tolist(),
        "classification_report": classification_report(y, preds, target_names=[idx_to_class[i] for i in sorted(idx_to_class.keys())], zero_division=0, output_dict=True)
    }
    return metrics

val_metrics = evaluate(model, X_val, y_val)
test_metrics = evaluate(model, X_test, y_test)

val_metrics, test_metrics["accuracy"]

({'accuracy': 0.7990743696137887,
  'f1_macro': 0.8006127575398599,
  'precision_macro': 0.8004097751489546,
  'recall_macro': 0.8008784830823034,
  'confusion_matrix': [[1915, 337, 117], [327, 1777, 186], [88, 204, 1315]],
  'classification_report': {'cats': {'precision': 0.8218884120171673,
    'recall': 0.8083579569438581,
    'f1-score': 0.8150670355394765,
    'support': 2369.0},
   'dogs': {'precision': 0.7666091458153581,
    'recall': 0.7759825327510917,
    'f1-score': 0.7712673611111112,
    'support': 2290.0},
   'wildlife': {'precision': 0.8127317676143387,
    'recall': 0.8182949595519602,
    'f1-score': 0.8155038759689922,
    'support': 1607.0},
   'accuracy': 0.7990743696137887,
   'macro avg': {'precision': 0.8004097751489546,
    'recall': 0.8008784830823034,
    'f1-score': 0.8006127575398599,
    'support': 6266.0},
   'weighted avg': {'precision': 0.7993374629017047,
    'recall': 0.7990743696137887,
    'f1-score': 0.7991718469230186,
    'support': 6266.0}}},
 0

In [ ]:
# Cell 9
config = {
    "model_name": MODEL_NAME,
    "feature_type": FEATURE_TYPE,
    "classifier": CLASSIFIER_NAME,
    "split_id": SPLIT_ID,
    "transform_id": TRANSFORM_ID,
    "hog_params": HOG_PARAMS,
    "img_size": list(IMG_SIZE),
    "run_id": RUN_ID,

    # new: approximation + classifier params
    "nystroem_params": NYSTROEM_PARAMS,
    "linearsvc_params": LINEARSVC_PARAMS,
}

metrics_payload = {
    "val": val_metrics,
    "test": test_metrics
}

model_path = RUN_DIR / "model.pkl"
config_path = RUN_DIR / "config.json"
metrics_path = RUN_DIR / "metrics.json"

with open(model_path, "wb") as f:
    pickle.dump(model, f)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

# report path already includes classifier name to prevent collision
with open(REPORT_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)


str(model_path), str(metrics_path), str(REPORT_METRICS_PATH)

('/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/ml_basic_features/hog_svm/run_20260305_152233/model.pkl',
 '/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/ml_basic_features/hog_svm/run_20260305_152233/metrics.json',
 '/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/reports/metrics/hog_svm_svm_rbf_approx_nystroem_linearsvc_metrics.json')

In [ ]:
# Cell 10
with mlflow.start_run(run_name=f"{MODEL_NAME}_{RUN_ID}"):
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("feature_type", FEATURE_TYPE)
    mlflow.log_param("classifier", CLASSIFIER_NAME)
    mlflow.log_param("split_id", SPLIT_ID)
    mlflow.log_param("transform_id", TRANSFORM_ID)
    mlflow.log_param("run_id", RUN_ID)

    # log approximation params
    mlflow.log_param("nystroem_n_components", NYSTROEM_PARAMS["n_components"])
    mlflow.log_param("nystroem_gamma", NYSTROEM_PARAMS["gamma"])
    mlflow.log_param("linearsvc_C", LINEARSVC_PARAMS["C"])
    mlflow.log_param("linearsvc_max_iter", LINEARSVC_PARAMS["max_iter"])

    mlflow.log_metric("val_accuracy", val_metrics["accuracy"])
    mlflow.log_metric("val_f1_macro", val_metrics["f1_macro"])
    mlflow.log_metric("test_accuracy", test_metrics["accuracy"])
    mlflow.log_metric("test_f1_macro", test_metrics["f1_macro"])

    mlflow.log_artifact(str(model_path))
    mlflow.log_artifact(str(config_path))
    mlflow.log_artifact(str(metrics_path))

In [ ]:
# Cell 11
with open(model_path, "rb") as f:
    loaded_model = pickle.load(f)

sample_X = X_test[:8]
sample_preds = loaded_model.predict(sample_X)
sample_pred_labels = [idx_to_class[int(i)] for i in sample_preds]
sample_pred_labels

['wildlife',
 'wildlife',
 'dogs',
 'wildlife',
 'wildlife',
 'dogs',
 'wildlife',
 'dogs']